# Handlers

> Route handlers, router initialization, and virtual collection wiring for the file browser.

In [ ]:
#| default_exp routes.handlers

In [ ]:
#| export
import inspect
from dataclasses import dataclass, field
from typing import Any, Callable, List, Optional, Tuple

from fasthtml.common import APIRouter

from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.routes.router import init_virtual_collection_router
from cjm_fasthtml_virtual_collection.components.table import render_cell_oob, render_visible_cells_oob

from cjm_file_discovery.core.models import FileInfo

from cjm_fasthtml_file_browser.core.config import (
    FileBrowserConfig, FileBrowserCallbacks, SelectionMode, FileColumn,
)
from cjm_fasthtml_file_browser.core.models import BrowserState
from cjm_fasthtml_file_browser.core.protocols import FileSystemProvider
from cjm_fasthtml_file_browser.components.browser import render_file_browser
from cjm_fasthtml_file_browser.components.item import create_file_cell_renderer
from cjm_fasthtml_file_browser.components.utils import sort_files, filter_files

## FileBrowserRouters

Return type for `init_router` — contains both the browser and virtual collection routers.

In [ ]:
#| export
def _no_selection_oobs(changed_paths: List[str]) -> Tuple:
    """Default no-op for render_selection_oobs."""
    return ()

def _no_update_selection_oobs(selected_paths: List[str], changed_paths: List[str]) -> Tuple:
    """Default no-op for update_selection_oobs."""
    return ()

def _no_current_path() -> str:
    """Default no-op for current_path."""
    return ""

def _no_sync_items() -> None:
    """Default no-op for sync_items."""
    pass

@dataclass
class FileBrowserRouters:
    """Return value from init_router — both routers, URL bundle, render, and OOB helpers."""
    browser: APIRouter                                   # File-specific routes (navigate, select, refresh, path_input)
    collection: APIRouter                                # Virtual collection routes (nav, focus, activate, sort, viewport)
    urls: VirtualCollectionUrls                          # URL bundle for rendering
    render: Callable                                     # () -> Any, renders the full file browser component
    render_selection_oobs: Callable = field(default=_no_selection_oobs)  # (changed_paths) -> Tuple, targeted checkbox OOBs
    update_selection_oobs: Callable = field(default=_no_update_selection_oobs)  # (selected_paths, changed_paths) -> Tuple, sync + OOBs
    current_path: Callable = field(default=_no_current_path)  # () -> str, current browsed directory path
    sync_items: Callable = field(default=_no_sync_items)  # () -> None, rebuild items from current browser state

In [ ]:
#| export
def _handle_navigate(
    provider: FileSystemProvider,           # File system provider
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    callbacks: Optional[FileBrowserCallbacks],  # Optional callbacks
    path: str,                              # Path to navigate to
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
) -> Any:  # Rendered browser component
    """Handle navigation to a new directory."""
    # Validate navigation if callback provided
    if callbacks and callbacks.validate_navigation:
        valid, error = callbacks.validate_navigation(path)
        if not valid:
            state = state_getter()
            return render_fn(state)
    
    # Normalize path
    normalized = provider.normalize_path(path)
    
    # Update state
    state = state_getter()
    state.current_path = normalized
    state_setter(state)
    
    # Call callback if provided
    if callbacks and callbacks.on_navigate:
        callbacks.on_navigate(normalized)
    
    return render_fn(state)

In [ ]:
#| export
def _handle_path_input(
    provider: FileSystemProvider,           # File system provider
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    callbacks: Optional[FileBrowserCallbacks],  # Optional callbacks
    path: str,                              # Path input by user
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
    navigate_fn: Callable[[str], Any],      # Function to handle navigation
) -> Any:  # Rendered browser component
    """Handle direct path input."""
    # Validate path exists and is directory
    valid, error = provider.is_valid_path(path)
    if valid and provider.is_directory(path):
        return navigate_fn(path)
    else:
        # Invalid path - stay on current
        state = state_getter()
        return render_fn(state)

## Selection Handlers

In [ ]:
#| export
def _handle_select(
    config: FileBrowserConfig,              # Browser configuration
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    callbacks: Optional[FileBrowserCallbacks],  # Optional callbacks
    path: str,                              # Path to select/deselect
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
) -> Any:  # Rendered browser component
    """Handle file selection."""
    # Validate selection if callback provided
    if callbacks and callbacks.validate_selection:
        valid, error = callbacks.validate_selection(path)
        if not valid:
            state = state_getter()
            return render_fn(state)
    
    state = state_getter()
    
    # Handle selection based on mode
    if config.selection_mode == SelectionMode.SINGLE:
        if state.selection.is_selected(path):
            state.selection.clear()
        else:
            state.selection.set_single(path)
    elif config.selection_mode == SelectionMode.MULTIPLE:
        state.selection.toggle(path)
        # Check max selections
        if config.max_selections and len(state.selection.selected_paths) > config.max_selections:
            # Remove oldest selection
            state.selection.selected_paths = state.selection.selected_paths[-config.max_selections:]
    
    state_setter(state)
    
    # Call callbacks
    if callbacks and callbacks.on_select:
        callbacks.on_select(path)
    if callbacks and callbacks.on_selection_change:
        callbacks.on_selection_change(state.selection.selected_paths)
    
    return render_fn(state)

In [ ]:
#| export
def _handle_refresh(
    state_getter: Callable[[], BrowserState],  # Function to get current state
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
) -> Any:  # Rendered browser component
    """Handle refresh (re-render with current state)."""
    state = state_getter()
    return render_fn(state)

## Selection Toggle Handler

In [ ]:
#| export
def _handle_toggle_select(
    item: FileInfo,                            # File item to toggle
    row_index: int,                            # Row index in VC items list
    config: FileBrowserConfig,                 # Browser configuration
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    callbacks: Optional[FileBrowserCallbacks],  # Optional callbacks
    sel_change_accepts_request: bool,          # Whether on_selection_change accepts request param
    columns: Tuple[ColumnDef, ...],            # VC column definitions
    vc_state: VirtualCollectionState,          # VC state (for total_items, cursor)
    vc_ids: VirtualCollectionHtmlIds,          # VC HTML IDs
    render_cell: Callable,                     # Cell renderer callback
    request: Any = None,                       # Optional HTMX request
) -> Any:  # Tuple of OOB elements (checkbox cell + callback extras)
    """Toggle file selection and return targeted OOB checkbox update."""
    browser_state = state_getter()
    if config.selection_mode == SelectionMode.SINGLE:
        if browser_state.selection.is_selected(item.path):
            browser_state.selection.clear()
        else:
            browser_state.selection.set_single(item.path)
    elif config.selection_mode == SelectionMode.MULTIPLE:
        browser_state.selection.toggle(item.path)
        if config.max_selections and len(browser_state.selection.selected_paths) > config.max_selections:
            browser_state.selection.selected_paths = browser_state.selection.selected_paths[-config.max_selections:]
    state_setter(browser_state)

    if callbacks and callbacks.on_select:
        callbacks.on_select(item.path)

    # Collect extra OOB elements from on_selection_change callback
    extra_oob = ()
    if callbacks and callbacks.on_selection_change:
        if sel_change_accepts_request and request is not None:
            result = callbacks.on_selection_change(browser_state.selection.selected_paths, request=request)
        else:
            result = callbacks.on_selection_change(browser_state.selection.selected_paths)
        if result and isinstance(result, tuple):
            extra_oob = result

    select_col = columns[0] if columns and columns[0].key == "select" else None
    if select_col:
        return (render_cell_oob(item, select_col, row_index, vc_state.total_items, vc_ids, render_cell),) + extra_oob
    return extra_oob

## Targeted OOB Helpers

In [ ]:
#| export
def _build_selection_oobs(
    changed_paths: List[str],              # File/folder paths whose checkbox state changed
    columns: Tuple[ColumnDef, ...],        # VC column definitions
    items: List[FileInfo],                 # Current directory items
    vc_state: VirtualCollectionState,      # VC state (for window bounds)
    vc_ids: VirtualCollectionHtmlIds,      # VC HTML IDs
    render_cell: Callable,                 # Cell renderer callback
) -> Tuple[Any, ...]:  # OOB cell elements for visible items only
    """Return OOB checkbox cell updates for changed paths visible in the browser."""
    select_col = columns[0] if columns and columns[0].key == "select" else None
    if not select_col:
        return ()

    path_to_index = {item.path: i for i, item in enumerate(items)}
    affected_indices = [path_to_index[p] for p in changed_paths if p in path_to_index]

    if not affected_indices:
        return ()

    return render_visible_cells_oob(
        column=select_col,
        item_indices=affected_indices,
        items=items,
        state=vc_state,
        ids=vc_ids,
        render_cell=render_cell,
    )


def _sync_and_build_selection_oobs(
    selected_paths: List[str],             # New full selection state (replaces browser selection)
    changed_paths: List[str],              # Paths whose checkbox state changed
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    columns: Tuple[ColumnDef, ...],        # VC column definitions
    items: List[FileInfo],                 # Current directory items
    vc_state: VirtualCollectionState,      # VC state (for window bounds)
    vc_ids: VirtualCollectionHtmlIds,      # VC HTML IDs
    render_cell: Callable,                 # Cell renderer callback
) -> Tuple[Any, ...]:  # OOB cell elements for visible changed items
    """Sync selection state into browser and return targeted checkbox OOBs."""
    browser_state = state_getter()
    browser_state.selection.selected_paths = list(selected_paths)
    state_setter(browser_state)
    return _build_selection_oobs(changed_paths, columns, items, vc_state, vc_ids, render_cell)

## Router Initialization

The `init_router` function creates both the browser-specific router and the virtual collection
router, managing internal VC state and the items list as closure state.

In [ ]:
#| export
def _build_columns(
    config: FileBrowserConfig,  # Browser config
) -> Tuple[ColumnDef, ...]:  # Column definitions for virtual collection
    """Build ColumnDef tuple from browser config, conditionally including checkbox column."""
    cols: List[ColumnDef] = []
    if config.selection_mode != SelectionMode.NONE:
        cols.append(ColumnDef(key="select", header=""))
    col_headers = {
        FileColumn.NAME: "Name",
        FileColumn.SIZE: "Size",
        FileColumn.MODIFIED: "Modified",
        FileColumn.TYPE: "Type",
        FileColumn.EXTENSION: "Extension",
        FileColumn.PATH: "Path",
    }
    sortable_cols = {FileColumn.NAME, FileColumn.SIZE, FileColumn.MODIFIED, FileColumn.TYPE}
    for fc in config.view.columns:
        cols.append(ColumnDef(
            key=fc.value,
            header=col_headers.get(fc, fc.value.title()),
            sortable=config.view.allow_sort_toggle and fc in sortable_cols,
        ))
    return tuple(cols)

In [ ]:
#| export
def _rebuild_items(
    provider: FileSystemProvider,  # File system provider
    state: BrowserState,           # Browser state with current_path
    config: FileBrowserConfig,     # Browser config (for filter/sort)
) -> List[FileInfo]:  # Filtered and sorted item list
    """List, filter, and sort directory contents."""
    listing = provider.list_directory(state.current_path)
    if listing.error:
        return []
    files = filter_files(listing.items, config.filter)
    return sort_files(files, sort_by=state.sort_by,
                      descending=state.sort_descending,
                      folders_first=config.view.sort_folders_first)

In [ ]:
#| export
def init_router(
    config: FileBrowserConfig,                              # Browser configuration
    provider: FileSystemProvider,                           # File system provider
    state_getter: Callable[[], BrowserState],               # Function to get current state
    state_setter: Callable[[BrowserState], None],           # Function to save state
    route_prefix: str = "/browser",                         # Route prefix for browser routes
    vc_route_prefix: str = "",                              # Route prefix for VC routes (auto: {route_prefix}/vc)
    callbacks: Optional[FileBrowserCallbacks] = None,       # Optional callbacks
    home_path: Optional[str] = None,                        # Home directory (defaults to provider)
) -> FileBrowserRouters:  # Browser + collection routers and URL bundle
    """Initialize file browser and virtual collection routers."""
    home = home_path or (provider.get_home_path() if hasattr(provider, 'get_home_path') else "/")
    _vc_prefix = vc_route_prefix or f"{route_prefix}/vc"

    # --- Internal mutable state (closure) ---
    columns = _build_columns(config)
    vc_config = VirtualCollectionConfig(
        prefix=config.vc_prefix or "",
        layout="table",
        columns=columns,
    )
    vc_state = VirtualCollectionState(visible_rows=1, cursor_index=0)
    vc_ids = VirtualCollectionHtmlIds(prefix=vc_config.prefix)
    vc_btn_ids = VirtualCollectionButtonIds(prefix=vc_config.prefix)

    _items: List[FileInfo] = []

    def _sync_items():
        """Rebuild items from current directory and update VC state."""
        browser_state = state_getter()
        _items[:] = _rebuild_items(provider, browser_state, config)
        vc_state.total_items = len(_items)
        vc_state.window_start = 0
        vc_state.cursor_index = 0 if _items else -1

    _sync_items()

    # Cell renderer — mutable ref so inner impl can be swapped after route registration
    _renderer_ref: list = [create_file_cell_renderer(
        config=config,
        get_selection=lambda: state_getter().selection,
    )]

    def render_cell(item, ctx):
        """Delegating render_cell that reads from mutable _renderer_ref."""
        return _renderer_ref[0](item, ctx)

    # --- Cache callback/setter signatures for request passing ---
    _sel_change_accepts_request = False
    if callbacks and callbacks.on_selection_change:
        sig = inspect.signature(callbacks.on_selection_change)
        _sel_change_accepts_request = 'request' in sig.parameters

    _setter_accepts_request = False
    _setter_sig = inspect.signature(state_setter)
    _setter_accepts_request = 'request' in _setter_sig.parameters

    def _call_state_setter(state: BrowserState, request=None):
        """Call state_setter, passing request if the setter accepts it."""
        if _setter_accepts_request and request is not None:
            state_setter(state, request=request)
        else:
            state_setter(state)

    # --- VC callbacks (thin wrappers calling extracted handlers) ---
    def _do_toggle_select(item, row_index, request=None):
        """Delegate to extracted _handle_toggle_select."""
        setter = (lambda s: _call_state_setter(s, request)) if _setter_accepts_request else state_setter
        return _handle_toggle_select(
            item, row_index, config, state_getter, setter,
            callbacks, _sel_change_accepts_request,
            columns, vc_state, vc_ids, render_cell, request,
        )

    def _do_navigate_oob(path: str, request=None) -> Tuple:
        """Navigate to path and return full browser as OOB self-swap."""
        normalized = provider.normalize_path(path)
        browser_state = state_getter()
        browser_state.current_path = normalized
        _call_state_setter(browser_state, request)
        if callbacks and callbacks.on_navigate:
            callbacks.on_navigate(normalized)
        _sync_items()
        browser_html = _render_browser_full()
        browser_html.attrs["hx-swap-oob"] = "outerHTML"
        return (browser_html,)

    def _on_activate(item, row_index, st, request=None):
        """Handle Enter/Space on focused row."""
        if item.is_directory:
            return _do_navigate_oob(item.path, request=request)
        elif config.can_select(item):
            return _do_toggle_select(item, row_index, request=request)
        return ()

    def _on_refocus(item, row_index, st, request=None):
        """Handle click on already-focused row."""
        if item.is_directory:
            return _do_navigate_oob(item.path, request=request)
        elif config.can_select(item):
            return _do_toggle_select(item, row_index, request=request)
        return ()

    def _sort_callback(items_list, column_key, ascending):
        """Sort items in place and update browser state."""
        browser_state = state_getter()
        browser_state.sort_by = column_key
        browser_state.sort_descending = not ascending
        _call_state_setter(browser_state)
        items_list[:] = sort_files(
            items_list, sort_by=column_key, descending=not ascending,
            folders_first=config.view.sort_folders_first,
        )

    # --- Virtual collection router ---
    vc_router, urls = init_virtual_collection_router(
        config=vc_config,
        state_getter=lambda: vc_state,
        state_setter=lambda s: None,
        get_items=lambda: _items,
        render_cell=render_cell,
        on_activate=_on_activate,
        on_refocus=_on_refocus,
        sort_callback=_sort_callback,
        route_prefix=_vc_prefix,
    )

    # --- Browser router ---
    browser_router = APIRouter(prefix=route_prefix)

    def _render_browser_full():
        """Render complete file browser."""
        browser_state = state_getter()
        listing = provider.list_directory(browser_state.current_path)
        return render_file_browser(
            items=_items, config=config, state=browser_state, listing=listing,
            vc_config=vc_config, vc_state=vc_state, vc_ids=vc_ids, vc_btn_ids=vc_btn_ids,
            urls=urls, render_cell=render_cell,
            navigate_url=navigate.to(), refresh_url=refresh.to(),
            path_input_url=path_input.to() if config.show_path_input else "",
            home_path=home,
        )

    def _do_navigate(path, request=None):
        """Navigate to path, rebuild items, re-render."""
        setter = (lambda s: _call_state_setter(s, request)) if _setter_accepts_request else state_setter
        return _handle_navigate(
            provider=provider, state_getter=state_getter, state_setter=setter,
            callbacks=callbacks, path=path,
            render_fn=lambda st: (_sync_items(), _render_browser_full())[-1],
        )

    @browser_router
    def navigate(request, path: str) -> Any:
        """Navigate to a new directory."""
        return _do_navigate(path, request=request)

    @browser_router
    def path_input(request, path: str) -> Any:
        """Handle direct path input."""
        setter = (lambda s: _call_state_setter(s, request)) if _setter_accepts_request else state_setter
        return _handle_path_input(
            provider=provider, state_getter=state_getter, state_setter=setter,
            callbacks=callbacks, path=path,
            render_fn=lambda st: (_sync_items(), _render_browser_full())[-1],
            navigate_fn=lambda p: _do_navigate(p, request=request),
        )

    @browser_router
    def refresh() -> Any:
        """Refresh the current directory listing."""
        _sync_items()
        return _render_browser_full()

    @browser_router
    def select(request, path: str) -> Any:
        """Toggle file selection via checkbox click (OOB cell update)."""
        for idx, item in enumerate(_items):
            if item.path == path:
                return _do_toggle_select(item, idx, request=request)
        return ()

    # Patch: replace inner renderer with one that has the real select URL
    _renderer_ref[0] = create_file_cell_renderer(
        config=config,
        get_selection=lambda: state_getter().selection,
        select_url=select.to(),
    )

    # --- Public helpers (thin wrappers calling extracted functions) ---
    def _render_selection_oobs(changed_paths):
        return _build_selection_oobs(changed_paths, columns, _items, vc_state, vc_ids, render_cell)

    def _update_selection_oobs(selected_paths, changed_paths):
        return _sync_and_build_selection_oobs(
            selected_paths, changed_paths, state_getter, state_setter,
            columns, _items, vc_state, vc_ids, render_cell,
        )

    def _get_current_path():
        return state_getter().current_path

    return FileBrowserRouters(
        browser=browser_router, collection=vc_router,
        urls=urls, render=_render_browser_full,
        render_selection_oobs=_render_selection_oobs,
        update_selection_oobs=_update_selection_oobs,
        current_path=_get_current_path,
        sync_items=_sync_items,
    )

In [ ]:
import tempfile
from pathlib import Path
from fasthtml.common import to_xml
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider

# Create test fixtures
with tempfile.TemporaryDirectory() as tmpdir:
    (Path(tmpdir) / "file1.txt").write_text("content1")
    (Path(tmpdir) / "file2.py").write_text("print('hi')")
    (Path(tmpdir) / "subdir").mkdir()

    config = FileBrowserConfig()
    provider = LocalFileSystemProvider()

    _state = BrowserState(current_path=tmpdir)
    def get_state(): return _state
    def set_state(s):
        global _state
        _state = s

    result = init_router(
        config=config,
        provider=provider,
        state_getter=get_state,
        state_setter=set_state,
        route_prefix="/browser",
    )

    # Verify return type
    assert isinstance(result, FileBrowserRouters)
    assert result.browser.prefix == "/browser"
    assert result.collection is not None
    assert result.urls.nav_up != ""
    assert result.urls.nav_down != ""
    assert result.urls.focus_row != ""
    assert result.urls.activate != ""
    assert result.urls.sort != ""

    # Verify render callable produces HTML
    assert callable(result.render)
    html = to_xml(result.render())
    assert "file-browser" in html
    assert "file1.txt" in html or "subdir" in html

    # Verify render_selection_oobs callable exists
    assert callable(result.render_selection_oobs)

print("Router initialization tests passed!")

In [ ]:
# Test render_selection_oobs — targeted checkbox OOB updates
with tempfile.TemporaryDirectory() as tmpdir:
    for i in range(5):
        (Path(tmpdir) / f"file{i}.txt").write_text(f"content{i}")

    config = FileBrowserConfig(selection_mode=SelectionMode.MULTIPLE, vc_prefix="test")
    provider = LocalFileSystemProvider()
    _state = BrowserState(current_path=tmpdir)
    def get_state(): return _state
    def set_state(s):
        global _state
        _state = s

    routers = init_router(
        config=config, provider=provider,
        state_getter=get_state, state_setter=set_state,
        route_prefix="/browser",
    )

    # Select file0 and file2 in browser state
    _state.selection.selected_paths = [str(Path(tmpdir) / "file0.txt"), str(Path(tmpdir) / "file2.txt")]

    # Get OOBs for paths in current directory
    oobs = routers.render_selection_oobs([str(Path(tmpdir) / "file0.txt"), str(Path(tmpdir) / "file2.txt")])
    assert len(oobs) > 0, "Should return OOB elements for visible items"
    for oob in oobs:
        oob_html = to_xml(oob)
        assert 'hx-swap-oob="outerHTML"' in oob_html
        assert 'col-select' in oob_html

    # Paths not in current directory return empty
    oobs_empty = routers.render_selection_oobs(["/nonexistent/path.txt"])
    assert oobs_empty == (), f"Expected empty tuple, got {len(oobs_empty)} elements"

    print("render_selection_oobs tests passed!")

In [ ]:
# Test update_selection_oobs — sync + render in one call
with tempfile.TemporaryDirectory() as tmpdir:
    for i in range(5):
        (Path(tmpdir) / f"file{i}.txt").write_text(f"content{i}")

    config = FileBrowserConfig(selection_mode=SelectionMode.MULTIPLE, vc_prefix="upd")
    provider = LocalFileSystemProvider()
    _state = BrowserState(current_path=tmpdir)
    def get_state(): return _state
    def set_state(s):
        global _state
        _state = s

    routers = init_router(
        config=config, provider=provider,
        state_getter=get_state, state_setter=set_state,
        route_prefix="/browser",
    )

    assert callable(routers.update_selection_oobs)

    # Initially no selection
    assert _state.selection.selected_paths == []

    # update_selection_oobs syncs selection AND returns OOBs
    file0 = str(Path(tmpdir) / "file0.txt")
    file2 = str(Path(tmpdir) / "file2.txt")
    oobs = routers.update_selection_oobs([file0, file2], [file0, file2])

    # Browser state should now have the selection synced
    assert file0 in _state.selection.selected_paths
    assert file2 in _state.selection.selected_paths

    # OOBs should be returned for visible items
    assert len(oobs) > 0
    for oob in oobs:
        oob_html = to_xml(oob)
        assert 'hx-swap-oob="outerHTML"' in oob_html
        assert 'col-select' in oob_html

    # Deselect file0 — pass new full selection, changed path is file0
    oobs2 = routers.update_selection_oobs([file2], [file0])
    assert file0 not in _state.selection.selected_paths
    assert file2 in _state.selection.selected_paths

    print("update_selection_oobs tests passed!")

In [ ]:
# Test current_path — returns browsed directory path
with tempfile.TemporaryDirectory() as tmpdir:
    config = FileBrowserConfig()
    provider = LocalFileSystemProvider()
    _state = BrowserState(current_path=tmpdir)
    def get_state(): return _state
    def set_state(s):
        global _state
        _state = s

    routers = init_router(
        config=config, provider=provider,
        state_getter=get_state, state_setter=set_state,
        route_prefix="/browser",
    )

    assert callable(routers.current_path)
    assert routers.current_path() == tmpdir

    # Simulate navigation by updating state
    _state.current_path = "/some/other/path"
    assert routers.current_path() == "/some/other/path"

    print("current_path tests passed!")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()